# LLaMA + RAG with Firecrawl Agentic Search (combined search-with-scraping demo)

This notebook demonstrates **Retrieval-Augmented Generation (RAG)** using:

- a local open-source LLM served via **Ollama** (for example, **LLaMA 3**),
- a lightweight vector database (**Chroma**) for retrieval, and
- a fully **Firecrawl-based agentic expansion loop** that uses the Firecrawl Search API with scraping enabled.

The logic is:

**seed URL scrape → local retrieval → if context is thin, Firecrawl search (with markdown scraping) → ingest new chunks → re-retrieve → grounded answer**

This notebook also removes DuckDuckGo and Trafilatura entirely.


## 0) Install dependencies

If you're in **Colab**, you may also want a terminal (`colab-xterm`) to run `ollama serve`.
If you're local, install Ollama normally and run the server in your terminal.


In [ ]:
# Core
!pip -q install -U ollama

# RAG stack (LangChain + Chroma)
!pip -q install -U langchain-community langchain-text-splitters langchain-chroma chromadb beautifulsoup4

# UI (optional)
!pip -q install -U gradio

# Firecrawl + env loading
!pip -q install -U firecrawl-py python-dotenv


### (Optional / Colab) Start Ollama server

If you're using Colab and want an in-notebook terminal:

1) run the next cell to enable `%xterm`  
2) in the xterm, start Ollama, for example:
   - `ollama serve`
   - then (once) `ollama pull llama3` and `ollama pull nomic-embed-text`


In [ ]:
# Optional: only needed on Colab
# !pip -q install colab-xterm
# %load_ext colabxterm
# %xterm


## 1) Configure models + persistence

- `LLM_MODEL`: the chat model used to generate answers
- `EMBED_MODEL`: the embedding model used for vector search
- `PERSIST_DIR`: where Chroma stores vectors (set to `None` for in-memory)
- `URL`: the seed page to index before agentic expansion starts

Set `FIRECRAWL_API_KEY` in `../../.env` or in your shell environment before running the notebook.


In [ ]:
LLM_MODEL = "llama3"
EMBED_MODEL = "nomic-embed-text"

# Where to persist the vector DB locally (recommended for iterative runs)
PERSIST_DIR = "../../data/vectorstores/chroma_db"   # set to None for in-memory

# Seed URL to index initially (agentic search can expand beyond this)
URL = "https://en.wikipedia.org/wiki/Ohiya"

# Optional: set FIRECRAWL_API_KEY in ../../.env or your shell environment


## 2) Quick smoke test: can we call the LLM?


In [ ]:
import ollama

resp = ollama.chat(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": "Say hello in one sentence and tell me you are ready for RAG."}]
)
print(resp["message"]["content"])


## 3) Build the initial vector store from the seed URL with Firecrawl scrape


In [ ]:
import os
from dotenv import load_dotenv
from firecrawl import Firecrawl
from langchain_core.documents import Document

# Splitter import (newer LangChain puts splitters in langchain_text_splitters)
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except Exception:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

# Chroma import (newer LangChain uses langchain_chroma)
try:
    from langchain_chroma import Chroma
except Exception:
    from langchain_community.vectorstores import Chroma

# Ollama embeddings import (community vs dedicated package depending on versions)
try:
    from langchain_ollama import OllamaEmbeddings
except Exception:
    from langchain_community.embeddings import OllamaEmbeddings

load_dotenv("../../.env")

api_key = os.getenv("FIRECRAWL_API_KEY")
if not api_key:
    raise ValueError("Set FIRECRAWL_API_KEY in ../../.env or your shell environment before running this notebook.")

firecrawl = Firecrawl(api_key=api_key)

def _get_field(obj, name, default=None):
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

scrape_result = firecrawl.scrape(URL, formats=["markdown"])
markdown = _get_field(scrape_result, "markdown")
metadata = _get_field(scrape_result, "metadata", {}) or {}

if not markdown:
    raise ValueError(f"Firecrawl returned no markdown for {URL}")

docs = [
    Document(
        page_content=markdown,
        metadata={
            "source": URL,
            "title": metadata.get("title") if isinstance(metadata, dict) else None,
            "provider": "firecrawl",
        },
    )
]

chunks = text_splitter.split_documents(docs)

vectorstore = Chroma(
    collection_name="rag_web_demo_agentic_firecrawl_combined",
    embedding_function=embeddings,
    persist_directory=PERSIST_DIR,
)

vectorstore.add_documents(chunks)

if hasattr(vectorstore, "persist"):
    try:
        vectorstore.persist()
    except Exception:
        pass

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
seen_urls = {URL}

print(f"Loaded seed URL via Firecrawl scrape: {URL}")
print(f"Seed docs: {len(docs)} | Seed chunks added: {len(chunks)}")


## 4) Agentic search + ingestion utilities

This notebook uses the **combined Firecrawl search-with-scraping path**. Instead of searching first and scraping later in two separate calls, it asks Firecrawl Search to return markdown content directly for the top search hits.


In [ ]:
def firecrawl_search_docs(query: str, limit: int = 5) -> list[Document]:
    try:
        results = firecrawl.search(
            query=query,
            limit=limit,
            sources=["web"],
            scrape_options={"formats": ["markdown"]},
        )
    except Exception as e:
        print(f"Firecrawl search failed: {e}")
        return []

    web_results = _get_field(results, "web", []) or []
    docs = []

    for r in web_results:
        url = _get_field(r, "url")
        title = _get_field(r, "title")
        markdown = _get_field(r, "markdown")

        if not url or not markdown or url in seen_urls:
            continue

        seen_urls.add(url)
        docs.append(
            Document(
                page_content=markdown,
                metadata={"source": url, "title": title, "provider": "firecrawl"},
            )
        )

    return docs

def ingest_search_results(query: str, limit: int = 3) -> int:
    new_docs = firecrawl_search_docs(query=query, limit=limit)
    if not new_docs:
        return 0

    new_chunks = text_splitter.split_documents(new_docs)
    vectorstore.add_documents(new_chunks)

    if hasattr(vectorstore, "persist"):
        try:
            vectorstore.persist()
        except Exception:
            pass

    return len(new_chunks)

def retrieve_docs(query: str):
    try:
        return retriever.get_relevant_documents(query)
    except Exception:
        return retriever.invoke(query)

def agentic_rag_answer(question: str, expand_if_needed: bool = True) -> tuple[str, list[str]]:
    # 1) Try local retrieval first
    retrieved = retrieve_docs(question)
    total_chars = sum(len(d.page_content) for d in retrieved)

    # 2) If context is thin, expand via Firecrawl search with markdown scraping enabled
    if expand_if_needed and total_chars < 1500:
        added = ingest_search_results(question, limit=3)
        if added:
            retrieved = retrieve_docs(question)

    context = "\n\n".join(d.page_content for d in retrieved)
    sources = []
    for d in retrieved:
        src = (d.metadata or {}).get("source")
        if src and src not in sources:
            sources.append(src)

    prompt = f"""You are a helpful assistant.
Use ONLY the context below to answer the question.
If the answer is not contained in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""

    resp = ollama.chat(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"], sources


## 5) Test it

Try a question that is **not fully covered** by the seed page to see the combined Firecrawl search-with-scraping path kick in.


In [ ]:
answer, sources = agentic_rag_answer("What is Ohiya? Give a short answer.")
print(answer)
print("\nSources:")
for s in sources:
    print("-", s)


## 6) Optional: Gradio UI

The UI shows the answer and the top sources used.


In [ ]:
import gradio as gr

def ui_fn(q):
    ans, srcs = agentic_rag_answer(q)
    return ans, "\n".join(f"- {s}" for s in srcs) if srcs else "(no sources)"

iface = gr.Interface(
    fn=ui_fn,
    inputs=gr.Textbox(lines=2, placeholder="Ask a question..."),
    outputs=[gr.Textbox(label="Answer"), gr.Textbox(label="Sources")],
    title="Agentic RAG (Ollama + Chroma + Firecrawl Combined Search + Scrape)",
    description=f"Seed URL indexed via Firecrawl scrape: {URL}\nChroma persist dir: {PERSIST_DIR}"
)

iface.launch(debug=False)
